# FlashSpec GPU measurement

This notebook measures TinyLlama-1.1B-Chat against a 160M drafter that uses the same 32,000-token vocabulary, plus a vanilla autoregressive baseline on the same prompts.

**An H100 is not required.** A free Colab Tesla T4 is enough. Colab Pro is useful only for the optional larger-model cell at the bottom, and only when that cell's memory check passes.

**Do not run this for a published number on the workstation Quadro P2000.** That GPU is Pascal, compute capability 6.1, with 5 GB. Current PyTorch CUDA wheels do not support it, and the probe cell below stops on purpose.

## Colab does not write this git repo

Files you see in the Colab file browser, and files on Google Drive, are not in the FlashSpec checkout. The last code cell builds `flashspec_bring_back.zip` and starts a browser download. On the workstation, unzip it and place the JSON files in `benchmarks/results/`. Commit those JSON files from the git checkout. A Drive copy is a spare, not the result.

## What to send back

The zip is the whole delivery. Inside it, these files matter:

1. `tinyllama_gpu_flashspec_ucb_<commit>.json`
2. `tinyllama_gpu_vanilla_ar_<commit>.json`
3. `BRING_BACK.txt`, which repeats the fields below so a quick look matches the JSON
4. `residual_colab_cpu.json`, only if the optional ASM cell printed `Wrote`
5. `tinyllama_throughput.png`, only if the plot cell printed a path. The plot is a convenience. The JSON is the measurement.

From each JSON, the fields that will be quoted are `hardware` (the real GPU name, Python, torch, CUDA), `git_commit`, `n_samples`, `metrics`, `unmeasured`, and `notes`.

If a cell raises, send the error text. Do not type a JSON file by hand. Do not copy the design-target table from the README into a file.

Prompts are random token ids. This run does not replace MT-Bench, HumanEval, or Alpaca, and it does not replace the Llama-3-8B design targets.


## 1. Get this version of the code

Set `BRANCH` to the git branch that contains this notebook and the float16 harness change. Leave `BRANCH` empty only when the notebook is already inside that checkout.

If the branch is not on GitHub yet, zip the checkout **without** `.venv`, upload the zip with the Colab files panel, and set `ZIP_PATH` to that path.

The install cell does not install PyTorch. Colab's existing CUDA build has to stay. If a later cell says CUDA disappeared, use Runtime → Restart session and do not `pip install torch`.


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

GIT_URL = "https://github.com/Mattral/FlashSpec"
BRANCH = ""  # example: "dev"
ZIP_PATH = ""  # example: "/content/flashspec.zip"

cwd = Path.cwd().resolve()
root = None
if (cwd / "pyproject.toml").is_file():
    root = cwd
elif ZIP_PATH:
    destination = Path("/content/flashspec_src")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(destination)
    candidates = [destination, *destination.glob("*/pyproject.toml")]
    for candidate in candidates:
        project = candidate if candidate.name != "pyproject.toml" else candidate.parent
        if (project / "pyproject.toml").is_file():
            root = project
            break
    if root is None:
        raise SystemExit(f"No pyproject.toml inside {ZIP_PATH}.")
elif BRANCH:
    root = Path("/content/FlashSpec")
    if not (root / "pyproject.toml").is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, GIT_URL, str(root)])
else:
    raise SystemExit(
        "This directory is not the FlashSpec checkout. Set BRANCH to the branch "
        "that contains notebooks/04_gpu_measurement.ipynb, or set ZIP_PATH to a "
        "zip of that checkout. Cloning main is wrong when this notebook is only "
        "on your machine."
    )

os.chdir(root)
print("Using", root)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu]", "pyyaml"]
)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA is not available after install. Restart the runtime. "
        "Do not pip install a CPU torch over Colab's CUDA torch."
    )


## 2. Record the GPU, then stop if it is too old

The next cell prints the only hardware string that may appear in the result. Compute capability below 7.0 stops here. That includes the Quadro P2000 (6.1). No JSON is written in that case.


In [ ]:
import torch

props = torch.cuda.get_device_properties(0)
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram_gb = props.total_memory / (1024 ** 3)
print(f"GPU: {name}")
print(f"Compute capability: {major}.{minor}")
print(f"Memory: {vram_gb:.2f} GB")
print(f"torch {torch.__version__}  CUDA {torch.version.cuda}")
if major < 7:
    raise SystemExit(
        f"{name} is compute capability {major}.{minor}. "
        "This notebook measures Turing or newer (T4 is 7.5). "
        "The Quadro P2000 is 6.1 and 5 GB, so it is not the target of this run. "
        "No result file was written."
    )
if vram_gb < 8:
    raise SystemExit(
        f"{name} reports {vram_gb:.2f} GB. TinyLlama float16 plus the 160M "
        "drafter wants about 8 GB. No result file was written."
    )
print("This GPU is in range for the TinyLlama pair. An H100 is not required.")


## 3. Run the paired measurement

`N_SAMPLES = 8` is the short run. If it finishes and the session is still open, set `N_SAMPLES = 30`, run this cell again, and send the 30-sample JSON. Both are usable. The file records which one you ran.

The drafter is `Felladrin/Llama-160M-Chat-v1` (vocab 32,000, same size family as TinyLlama). If the vocab check fails, the harness writes nothing. Send the error instead of picking a different model silently.

One drafter is loaded, so the bandit cannot adapt. This cell measures speculative decoding with a fixed drafter against vanilla autoregressive decoding.


In [ ]:
import tempfile
from pathlib import Path

import torch
import yaml

from benchmarks.harness import run_config_file

N_SAMPLES = 8
MAX_NEW_TOKENS = 32
N_WARMUP = 1

source = Path("benchmarks/configs/tinyllama_gpu.yaml")
config = yaml.safe_load(source.read_text(encoding="utf-8"))
config["n_samples"] = N_SAMPLES
config["max_new_tokens"] = MAX_NEW_TOKENS
config["n_warmup"] = N_WARMUP
config["dtype"] = "float16"
config["device"] = "cuda:0"

runtime_dir = Path(tempfile.mkdtemp(prefix="flashspec-gpu-"))
runtime_yaml = runtime_dir / "tinyllama_gpu.yaml"
runtime_yaml.write_text(yaml.safe_dump(config), encoding="utf-8")
output_dir = Path("benchmarks/results")
torch.cuda.empty_cache()
written = run_config_file(runtime_yaml, output_dir)
print("Wrote:")
for path in written:
    print(path.resolve())


## 4. Print the fields to bring back

Read this output before you close the session. It is a view of the JSON. If a number here disagrees with the file, the file wins.


In [ ]:
import json
from pathlib import Path

from scripts.update_benchmark_results import _validate_result_file

paths = sorted(Path("benchmarks/results").glob("tinyllama_gpu_*.json"))
if len(paths) < 2:
    raise SystemExit(
        "Expected two tinyllama_gpu JSON files. The run cell did not write them. "
        "Send the error from that cell. Do not create the files yourself."
    )
lines = []
for path in paths:
    errors = _validate_result_file(path)
    if errors:
        raise SystemExit("Schema errors:\n" + "\n".join(errors))
    document = json.loads(path.read_text(encoding="utf-8"))
    metrics = document["metrics"]
    lines.append(f"FILE {path.name}")
    lines.append(f"  hardware={document['hardware']}")
    lines.append(f"  git_commit={document['git_commit']}")
    lines.append(f"  model={document['model']} method={document['method']} gamma={document['gamma']}")
    lines.append(f"  n_samples={document['n_samples']} dataset={document['dataset']}")
    lines.append(f"  metrics={json.dumps(metrics, sort_keys=True)}")
    lines.append(f"  unmeasured={document['unmeasured']}")
    lines.append(f"  notes={document['notes']}")
report = "\n".join(lines)
Path("benchmarks/results/BRING_BACK.txt").write_text(report + "\n", encoding="utf-8")
print(report)
print()
print("Copy the JSON files into the git checkout under benchmarks/results/.")
print("Google Drive is not the repo. Commit only after the files are in the checkout.")


## 5. Optional bar chart

Skip this cell if you want. The PNG is not required. Do not treat it as a paper figure until the JSON is committed and reviewed.


In [ ]:
import json
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    import matplotlib.pyplot as plt

rows = []
gpu_name = "gpu"
for path in sorted(Path("benchmarks/results").glob("tinyllama_gpu_*.json")):
    document = json.loads(path.read_text(encoding="utf-8"))
    gpu_name = str(document["hardware"].get("gpu", gpu_name))
    mean = document["metrics"].get("tokens_per_second_mean")
    if isinstance(mean, (int, float)):
        rows.append((document["method"], float(mean)))
if len(rows) < 2:
    raise SystemExit("No throughput numbers to plot. The JSON is still the result.")
labels, values = zip(*rows)
figure, axis = plt.subplots(figsize=(6, 3.5))
axis.bar(labels, values)
axis.set_ylabel("tokens per second")
axis.set_title(f"TinyLlama pair on {gpu_name}\n(synthetic token ids, see JSON)")
figure.tight_layout()
destination = Path("benchmarks/results/tinyllama_throughput.png")
figure.savefig(destination, dpi=120)
print("Wrote", destination.resolve())
print("Optional. Bring it back only together with the two JSON files.")


## 6. Optional CPU assembly residual

Colab's CPU is Linux x86-64, so this is the first place the AVX2 residual kernel can load for this project. It is a CPU microbench, not the TinyLlama throughput number. If `pip install` or the import fails, the cell prints the error and writes nothing. Leave the Windows file `residual_cpu.json` untouched. This cell uses the name `residual_colab_cpu.json`.


In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

destination = Path("benchmarks/results/residual_colab_cpu.json")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    print("Skipped. flashspec-asm is Linux x86-64 only. No ASM file written.")
else:
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[asm]"],
        check=False,
    )
    if completed.returncode != 0:
        print("ASM extra failed to install. No ASM file written.")
    else:
        measured = subprocess.run(
            [
                sys.executable,
                "benchmarks/bench_residual.py",
                "--output",
                str(destination),
                "--iterations",
                "30",
            ],
            check=False,
        )
        if measured.returncode != 0 or not destination.is_file():
            print("Residual bench failed. No ASM file to bring back.")
            destination.unlink(missing_ok=True)
        else:
            print("Wrote", destination.resolve())
            print("Bring this file back as benchmarks/results/residual_colab_cpu.json.")
            print("Do not rename it to residual_cpu.json. That name is the Windows run.")


## 7. Download the zip

Run this cell last. In Colab it calls `files.download`. On a machine that is not Colab it prints the zip path so you can copy it yourself.

After the download lands on the workstation:

1. Unzip it.
2. Copy every `.json` into `benchmarks/results/` of the git checkout.
3. Leave `BRING_BACK.txt` and the PNG out of the commit unless you want them as notes. The JSON is what gets quoted.
4. `git status` should show the new JSON as untracked files that are not ignored.
5. Commit from the workstation. Do not commit from the Colab VM, and do not treat a Drive folder as the commit.


In [ ]:
import zipfile
from pathlib import Path

results = Path("benchmarks/results")
wanted = [
    *sorted(results.glob("tinyllama_gpu_*.json")),
    *sorted(results.glob("larger_gpu_*.json")),
    results / "BRING_BACK.txt",
    results / "tinyllama_throughput.png",
    results / "residual_colab_cpu.json",
]
present = [path for path in wanted if path.is_file()]
json_files = [path for path in present if path.name.startswith("tinyllama_gpu_")]
if len(json_files) < 2:
    raise SystemExit(
        "The two TinyLlama JSON files are missing, so there is nothing to download. "
        "Re-run the measurement cell. Do not zip an empty result."
    )
zip_path = Path("flashspec_bring_back.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in present:
        archive.write(path, arcname=path.name)
print("Zip contains:")
for path in present:
    print(" ", path.name)
print("Workstation destination for the JSON files: benchmarks/results/")
try:
    from google.colab import files
    files.download(str(zip_path))
    print("Browser download started. Drive is not the repo.")
except ImportError:
    print("Not Colab. Copy this file yourself:")
    print(zip_path.resolve())


## 8. Optional Llama-3 run

Leave this cell alone for the TinyLlama delivery.

Run it only when the probe printed at least 24 GB and compute capability 8.0 or newer (A100, L4 24 GB is borderline, H100 is fine and still not required). A T4 cannot hold Llama-3-8B in bfloat16.

Set both ids yourself. They must share a vocabulary. TinyLlama's 32,000-token drafter does not match Llama-3's 128,256-token vocabulary, and the harness will refuse that pair without writing a file. A gated model needs `HF_TOKEN` in the Colab secrets. This cell does not download anything until you set `RUN_LARGER_MODEL = True`.


In [ ]:
import tempfile
from pathlib import Path

import torch
import yaml

from benchmarks.harness import run_config_file

RUN_LARGER_MODEL = False
LARGER_TARGET = ""  # example: "meta-llama/Llama-3.2-3B-Instruct"
LARGER_DRAFTER = ""  # must use the same tokenizer vocabulary as the target

if not RUN_LARGER_MODEL:
    print("Skipped. The TinyLlama JSON is the required bring-back. H100 is not required.")
else:
    props = torch.cuda.get_device_properties(0)
    major, _minor = torch.cuda.get_device_capability(0)
    vram_gb = props.total_memory / (1024 ** 3)
    if major < 8 or vram_gb < 24:
        raise SystemExit(
            f"{torch.cuda.get_device_name(0)} has capability {major}.x and "
            f"{vram_gb:.1f} GB. Refusing the larger-model run. No file written."
        )
    if not LARGER_TARGET or not LARGER_DRAFTER:
        raise SystemExit("Set LARGER_TARGET and LARGER_DRAFTER. No file written.")
    config = {
        "model": LARGER_TARGET,
        "drafter": LARGER_DRAFTER,
        "device": "cuda:0",
        "dtype": "bfloat16",
        "gamma": 4,
        "batch_size": 1,
        "max_new_tokens": 32,
        "n_warmup": 1,
        "n_samples": 4,
        "prompt_length": 16,
        "seed": 0,
        "dataset": "synthetic_token_ids",
        "temperature": 1.0,
        "methods": ["flashspec_ucb", "vanilla_ar"],
    }
    runtime_dir = Path(tempfile.mkdtemp(prefix="flashspec-larger-"))
    runtime_yaml = runtime_dir / "larger_gpu.yaml"
    runtime_yaml.write_text(yaml.safe_dump(config), encoding="utf-8")
    written = run_config_file(runtime_yaml, Path("benchmarks/results"))
    print("Wrote:")
    for path in written:
        print(path.resolve())
    print("Run the download cell again. It also packs larger_gpu_*.json.")
    print("Those files belong in benchmarks/results/ next to the TinyLlama JSON.")
